python scripts/reformat_sabin.py --datadir data --rename data/rename_columns.xlsx --correction data/fix_values.xlsx --output combined_sabin.tsv

In [ ]:
python scripts/reformat_sabin.py --datadir data --rename data/rename_columns.xlsx --correction data/fix_values.xlsx --output combined_sabin.tsv --cache results/combined_db

In [1]:
import pandas as pd
import os
import re

def load_table(file, separator=None):
    df = ''
    if str(file).split('.')[-1] == 'tsv':
        separator = '\t' if separator is None else separator
        df = pd.read_csv(file, encoding='latin-1', sep=separator, dtype='str')
    elif str(file).split('.')[-1] == 'csv':
        separator = ',' if separator is None else separator
        df = pd.read_csv(file, encoding='latin-1', sep=separator, dtype='str')
    elif str(file).split('.')[-1] in ['xls', 'xlsx']:
        df = pd.read_excel(file, index_col=None, header=0, sheet_name=0, dtype='str')
        df.fillna('', inplace=True)
    else:
        print('Wrong file format. Compatible file formats: TSV, CSV, XLS, XLSX')
        exit()
    return df

In [5]:
df_sabin_covid = pd.read_excel(
    './../data/SABIN/20230823_SABIN_Covid_2023ateSE33.xlsx'
)

df_dict = pd.read_excel(
    './../data/SABIN/20230823_SABIN_Painel respiratorio 2023ateSE33.xlsx', 
    index_col=None, header=0, 
    sheet_name=None, 
)
df_paineis = pd.DataFrame()
for sheet in df_dict.keys():
    df_paineis = df_paineis.append(df_dict[sheet].assign(ExcelSheet=sheet))

/tmp/ipykernel_3232/1232156733.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_paineis = df_paineis.append(df_dict[sheet].assign(ExcelSheet=sheet))
/tmp/ipykernel_3232/1232156733.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_paineis = df_paineis.append(df_dict[sheet].assign(ExcelSheet=sheet))
/tmp/ipykernel_3232/1232156733.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_paineis = df_paineis.append(df_dict[sheet].assign(ExcelSheet=sheet))
/tmp/ipykernel_3232/1232156733.py:12: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_paineis = df_paineis.append(df_dict[sheet].assign(ExcelSheet=sheet))


In [141]:
df_full = pd.concat([df_sabin_covid, df_paineis], axis=0)
df_full.head()

,OS,Código Posto,Estado,Municipio,DataAtendimento,DataNascimento,Sexo,Descricao,Parametro,Resultado,DataAssinatura,ExcelSheet
0,321-66464-12445,321,DISTRITO FEDERAL,BRASÍLIA,21/12/2022,17/11/1980,Feminino,TESTE MOLECULAR PARA DETECÇÃO DO CORONAVÍRUS S...,NALVO,NÃO DETECTADO (Ausência de RNA específico de S...,2023-01-01 00:00:00,NaN
1,525-66472-8612,525,DISTRITO FEDERAL,BRASÍLIA,29/12/2022,26/11/1983,Feminino,TESTE MOLECULAR PARA DETECÇÃO DO CORONAVÍRUS S...,NALVO,DETECTADO (Presença de RNA específico de SARS-...,2023-01-01 00:00:00,NaN
2,324-66472-9108,324,PARÁ,BELÉM,29/12/2022,21/09/1958,Masculino,TESTE MOLECULAR PARA DETECÇÃO DO CORONAVÍRUS S...,NALVO,NÃO DETECTADO (Ausência de RNA específico de S...,2023-01-01 00:00:00,NaN
3,891-66472-10158,891,GOIAS,ANAPOLIS,29/12/2022,17/11/2002,Masculino,TESTE MOLECULAR PARA DETECÇÃO DO CORONAVÍRUS S...,NALVO,DETECTADO (Presença de RNA específico de SARS-...,2023-01-01 00:00:00,NaN
4,407-66472-10869,407,TOCANTINS,PALMAS,29/12/2022,17/09/1959,Feminino,TESTE MOLECULAR PARA DETECÇÃO DO CORONAVÍRUS S...,NALVO,DETECTADO (Presença de RNA específico de SARS-...,2023-01-01 00:00:00,NaN


In [124]:
df_full.query("OS == '743-66605-21030'")

,OS,Código Posto,Estado,Municipio,DataAtendimento,DataNascimento,Sexo,Descricao,Parametro,Resultado,DataAssinatura,ExcelSheet,DataAtendimentoFixed
54789,743-66605-21030,743,DISTRITO FEDERAL,BRASÍLIA,2023-11-05 00:00:00,31/05/1979,Masculino,TESTE MOLECULAR PARA DETECÇÃO DO CORONAVÍRUS S...,NALVO,DETECTADO (Presença de RNA específico de SARS-...,2023-12-05 00:00:00,NaN,2023-11-05 00:00:00


In [122]:
df_full.query("DataAtendimentoFixed > '2023-12'")

,OS,Código Posto,Estado,Municipio,DataAtendimento,DataNascimento,Sexo,Descricao,Parametro,Resultado,DataAssinatura,ExcelSheet,DataAtendimentoFixed
5795,743-66486-58,743,DISTRITO FEDERAL,BRASÍLIA,2023-12-01 00:00:00,2004-03-08 00:00:00,Feminino,DETECCAO QUALITATIVA DE ANTIGENOS DE SARS-CoV-2,COVIDECO,NÃO DETECTADO,2023-12-01 00:00:00,NaN,2023-12-01 00:00:00
5796,939-66486-135,939,DISTRITO FEDERAL,BRASÍLIA,2023-12-01 00:00:00,1994-03-05 00:00:00,Masculino,TESTE MOLECULAR PARA DETECÇÃO DO CORONAVÍRUS S...,NALVO,NÃO DETECTADO (Ausência de RNA específico de S...,2023-12-01 00:00:00,NaN,2023-12-01 00:00:00
5797,939-66486-150,939,DISTRITO FEDERAL,BRASÍLIA,2023-12-01 00:00:00,25/02/1962,Feminino,DETECCAO QUALITATIVA DE ANTIGENOS DE SARS-CoV-2,COVIDECO,NÃO DETECTADO,2023-12-01 00:00:00,NaN,2023-12-01 00:00:00
5798,1417-66486-158,1417,DISTRITO FEDERAL,BRASÍLIA,2023-12-01 00:00:00,22/06/1982,Feminino,TESTE MOLECULAR PARA SARS-CoV-2 (COVID 19) COM...,TMR19RES1,DETECTADO (Presença de RNA específico de SARS-...,2023-12-01 00:00:00,NaN,2023-12-01 00:00:00
5799,013-66486-691,13,DISTRITO FEDERAL,BRASÍLIA,2023-12-01 00:00:00,18/09/1971,Masculino,TESTE MOLECULAR PARA DETECÇÃO DO CORONAVÍRUS S...,NALVO,NÃO DETECTADO (Ausência de RNA específico de S...,2023-12-01 00:00:00,NaN,2023-12-01 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11390,791-66698-13475,791,MINAS GERAIS,UBERLANDIA,2023-12-08 00:00:00,25/06/2023,Masculino,"PAINEL PARA VÍRUS SARS-CoV-2, INFLUENZA A, B E...",PCRVRESPBM4,NÃO DETECTADO (Ausência de RNA viral específico),16/08/2023,PCRVRESP,2023-12-08 00:00:00
11391,791-66698-13475,791,MINAS GERAIS,UBERLANDIA,2023-12-08 00:00:00,25/06/2023,Masculino,"PAINEL PARA VÍRUS SARS-CoV-2, INFLUENZA A, B E...",PCRVRESP,NÃO DETECTADO (Ausência de RNA viral específico),16/08/2023,PCRVRESP,2023-12-08 00:00:00
11392,791-66698-13475,791,MINAS GERAIS,UBERLANDIA,2023-12-08 00:00:00,25/06/2023,Masculino,"PAINEL PARA VÍRUS SARS-CoV-2, INFLUENZA A, B E...",GENES,NÃO DETECTADO,16/08/2023,PCRVRESP,2023-12-08 00:00:00
11393,791-66698-13475,791,MINAS GERAIS,UBERLANDIA,2023-12-08 00:00:00,25/06/2023,Masculino,"PAINEL PARA VÍRUS SARS-CoV-2, INFLUENZA A, B E...",GENERDRP,NÃO DETECTADO,16/08/2023,PCRVRESP,2023-12-08 00:00:00


Data Sabin

Análise do arquivo - 20230823_SABIN_Painel respiratorio 2023ateSE33.xlsx

se a gente considera xxxx-xx-xx como yyyy-mm-dd, então dá problema de data futura

se a gente considera xxxx-xx-xx como dd-mm-yyyy, então dá problema de 'shift' nos dados

In [78]:
import re

In [85]:
df_paineis.head()[['OS','DataAtendimento', 'ExcelSheet']]

,OS,DataAtendimento,ExcelSheet
0,565-66476-5360,2023-02-01 00:00:00,PAINCOVI
1,565-66476-5360,2023-02-01 00:00:00,PAINCOVI
2,565-66476-5360,2023-02-01 00:00:00,PAINCOVI
3,565-66476-5360,2023-02-01 00:00:00,PAINCOVI
4,565-66476-5360,2023-02-01 00:00:00,PAINCOVI


In [13]:
# ajuste técinico
df_combined_sabin = pd.read_csv('../combined_sabin.tsv', sep='\t')
df_combined_sabin_snakemake = pd.read_csv('../results/combined_sabin.tsv', sep='\t')

/tmp/ipykernel_2942/3932633123.py:3: DtypeWarning: Columns (1,10,23) have mixed types. Specify dtype option on import or set low_memory=False.
  df_combined_sabin_snakemake = pd.read_csv('../results/combined_sabin.tsv', sep='\t')


## Duplicatas

In [3]:
df_duplicates = (
    df_combined
    .assign( one=1 )
    .groupby(['sample_id'])
    .agg(num_duplicates=('one', 'sum'))
    .query("num_duplicates > 1")
    .assign(num_duplicates=lambda x: x['num_duplicates']-1)
)


print( df_duplicates['num_duplicates'].sum(), df_duplicates['num_duplicates'].max(), df_duplicates['num_duplicates'].min() )
df_duplicates.sort_values('num_duplicates', ascending=False).head(10)

0 nan nan


,num_duplicates
sample_id,


In [4]:
df_duplicates = (
    df_combined
    # .query("test_kit == 'unknown'")
    .assign( one=1 )
    .groupby(['test_id'])
    .agg(num_duplicates=('one', 'sum'))
    .query("num_duplicates > 1")
    .assign(num_duplicates=lambda x: x['num_duplicates']-1)
)


print( df_duplicates['num_duplicates'].sum(), df_duplicates['num_duplicates'].max(), df_duplicates['num_duplicates'].min() )
df_duplicates.sort_values('num_duplicates', ascending=False).head(10)

2378 1 1


,num_duplicates
test_id,
001-66156-14273,1
463-66562-17592,1
463-66667-16115,1
466-66434-18608,1
466-66434-18901,1
466-66639-18951,1
474-66173-8412,1
474-66268-14113,1
482-66154-19237,1


In [20]:
df_os_multiple_tests = (
    df_combined
    .groupby('test_id')
    .agg(kits=('test_kit',lambda x: ','.join(x.tolist())))
    .query("kits.str.contains(',')", engine='python') 
)

In [21]:
df_os_multiple_tests.head(10)

,kits
test_id,
001-66156-14273,"covid_antigen,covid_pcr"
001-66156-14280,"covid_antigen,covid_pcr"
001-66156-14376,"covid_antigen,covid_pcr"
001-66166-12255,"covid_antigen,covid_pcr"
001-66266-18643,"covid_antigen,covid_pcr"
001-66270-16315,"covid_antigen,covid_pcr"
001-66272-1706,"covid_antigen,covid_pcr"
001-66276-5891,"covid_antigen,covid_pcr"
001-66289-7819,"covid_antigen,covid_pcr"


In [22]:
df_combined.columns

Index(['lab_id', 'test_id', 'test_kit', 'patient_id', 'sample_id', 'state',
       'location', 'date_testing', 'epiweek', 'age', 'sex', 'FLUA_test_result',
       'Ct_FluA', 'FLUB_test_result', 'Ct_FluB', 'VSR_test_result', 'Ct_VSR',
       'SC2_test_result', 'Ct_geneE', 'Ct_geneN', 'Ct_geneS', 'Ct_ORF1ab',
       'Ct_RDRP', 'geneS_detection', 'META_test_result', 'RINO_test_result',
       'PARA_test_result', 'ADENO_test_result', 'BOCA_test_result',
       'COVS_test_result', 'ENTERO_test_result', 'BAC_test_result'],
      dtype='object')

## Uniqueness

In [5]:
if 'unknown' in df_combined['test_kit'].unique():
    print('WARNING! unknown test kits found')

df_combined['test_kit'].unique()

array(['covid_pcr', 'covid_antigen', 'test_4', 'test_21', 'test_24'],
      dtype=object)

In [6]:
df_combined['state'].unique()

array(['Distrito Federal', 'Bahia', 'Goiás', 'São Paulo', 'Minas Gerais',
       'Paraná', 'Mato Grosso do Sul', 'Tocantins', 'Roraima', nan,
       'Amazonas', 'Santa Catarina', 'Pará', 'Mato Grosso'], dtype=object)

In [7]:
df_combined['location'].unique()

array(['BRASÍLIA', 'BARREIRAS', 'FORMOSA', 'RIBEIRÃO PRETO', 'UBERLANDIA',
       'MARINGA', 'RIO DE JANEIRO', 'UBERABA', 'DOURADOS', 'PALMAS',
       'BOA VISTA', 'LONDRINA', 'CAMPINAS', nan, 'SALVADOR', 'MANAUS',
       'FLORIANOPOLIS', 'ANAPOLIS', 'SAO CAETANO', 'BELÉM',
       'SÃO JOSE DOS CAMPOS', 'CUIABA', 'LUZIANIA', 'TANGARA DA SERRA',
       'VINHEDO', 'CRISTALINA', 'PLANALTINA', 'PADRE BERNARDO',
       'SANTO ANTÔNIO DO DESCOBERTO'], dtype=object)

In [8]:
df_combined['date_testing'].max(), df_combined['date_testing'].min()

('2023-08-20', '2020-01-11')

In [9]:
df_combined['age'].unique(), df_combined['age'].min(), df_combined['age'].max()

(array([ 64,   4,  39,  38,  41,  56,  25,  67,  36,  88,  47,  40,  26,
         44,   3,  62,  31,  32,  23,  42,  71,  37,  74,  51,  52,  35,
         77,  10,  50,  78,  34,  43,  80,  29,  18,   6,   1,  76,  85,
         14,  19,  48,  61,  17,  13,  24,  59,  55,  63,  73,  58,  27,
         72,  57,  30,  65,  33,  46,  28,  60,  16,  53,  45,   8,  22,
         49,  79,  15,  20,  54,  21,  66,  68,  90,  69,  83,   2,   5,
         12,  86,  11,   7,   9,  75,  87,  91,  70,   0,  82,  95,  97,
         84,  81,  93, 100,  89,  92,  94,  99,  96,  98, 102, 103, 101,
        106, 122, 123, 105, 104, 128,  -1, 136, 107]),
 -1,
 136)

In [10]:
df_combined['sex'].unique()

array(['M', 'F', 'N', 'I', nan], dtype=object)

In [11]:
for column in df_combined.columns:
    if column.endswith('_result'):
        print(column, df_combined[column].unique())

FLUA_test_result ['NT' 'Neg' 'Pos']
FLUB_test_result ['NT' 'Neg' 'Pos']
VSR_test_result ['NT' 'Neg' 'Pos']
SC2_test_result ['Neg' 'Pos' 'NT']
META_test_result ['NT' 'Neg' 'Pos']
RINO_test_result ['NT' 'Neg' 'Pos']
PARA_test_result ['NT' 'Neg' 'Pos']
ADENO_test_result ['NT' 'Neg' 'Pos']
BOCA_test_result ['NT' 'Neg' 'Pos']
COVS_test_result ['NT' 'Neg' 'Pos']
ENTERO_test_result ['NT' 'Neg' 'Pos']
BAC_test_result ['NT' 'Neg' 'Pos']


In [12]:
for col in df_combined.columns:
    if col.endswith('_result'):
        print(col)

FLUA_test_result
FLUB_test_result
VSR_test_result
SC2_test_result
META_test_result
RINO_test_result
PARA_test_result
ADENO_test_result
BOCA_test_result
COVS_test_result
ENTERO_test_result
BAC_test_result


## Inconsistencies

In [23]:
df_combined.columns

Index(['lab_id', 'test_id', 'test_kit', 'patient_id', 'sample_id', 'state',
       'location', 'date_testing', 'epiweek', 'age', 'sex', 'FLUA_test_result',
       'Ct_FluA', 'FLUB_test_result', 'Ct_FluB', 'VSR_test_result', 'Ct_VSR',
       'SC2_test_result', 'Ct_geneE', 'Ct_geneN', 'Ct_geneS', 'Ct_ORF1ab',
       'Ct_RDRP', 'geneS_detection', 'META_test_result', 'RINO_test_result',
       'PARA_test_result', 'ADENO_test_result', 'BOCA_test_result',
       'COVS_test_result', 'ENTERO_test_result', 'BAC_test_result'],
      dtype='object')

Verificando se um mesmo test_id + test_kit apresentam mais de um resultado para um mesmo patógeno

In [24]:
agg_test_rules = {
    col+'_kit': (col, lambda x: ','.join(x.tolist()))
    for col in df_combined.columns
    if col.endswith('_result')
}

df_os_non_contraditory_test_results = (
    df_combined
    .groupby(['test_id', 'test_kit'])
    .agg(**agg_test_rules)
)

In [25]:
for col in df_os_non_contraditory_test_results.columns:
    if col.endswith('_kit'):
        print(col, df_os_non_contraditory_test_results[col].unique())
        # Devem ser apenas ['NT' 'Neg' 'Pos']

FLUA_test_result_kit ['NT' 'Neg' 'Pos']
FLUB_test_result_kit ['NT' 'Neg' 'Pos']
VSR_test_result_kit ['NT' 'Neg' 'Pos']
SC2_test_result_kit ['Pos' 'Neg' 'NT']
META_test_result_kit ['NT' 'Neg' 'Pos']
RINO_test_result_kit ['NT' 'Neg' 'Pos']
PARA_test_result_kit ['NT' 'Neg' 'Pos']
ADENO_test_result_kit ['NT' 'Neg' 'Pos']
BOCA_test_result_kit ['NT' 'Neg' 'Pos']
COVS_test_result_kit ['NT' 'Neg' 'Pos']
ENTERO_test_result_kit ['NT' 'Neg' 'Pos']
BAC_test_result_kit ['NT' 'Neg' 'Pos']


In [26]:
df_os_non_contraditory_test_results.query("SC2_test_result_kit == 'Neg,Pos'")

,,FLUA_test_result_kit,FLUB_test_result_kit,VSR_test_result_kit,SC2_test_result_kit,META_test_result_kit,RINO_test_result_kit,PARA_test_result_kit,ADENO_test_result_kit,BOCA_test_result_kit,COVS_test_result_kit,ENTERO_test_result_kit,BAC_test_result_kit
test_id,test_kit,,,,,,,,,,,,


In [27]:
df_os_non_contraditory_test_results.reset_index().query("test_kit == 'test_4'").query("FLUA_test_result_kit == 'Neg,Neg'")

,test_id,test_kit,FLUA_test_result_kit,FLUB_test_result_kit,VSR_test_result_kit,SC2_test_result_kit,META_test_result_kit,RINO_test_result_kit,PARA_test_result_kit,ADENO_test_result_kit,BOCA_test_result_kit,COVS_test_result_kit,ENTERO_test_result_kit,BAC_test_result_kit


In [ ]:
gene_df.query("requisicao == '1002751831'")

,requisicao,data,data_de_nascimento,idade,sexo,teste,data_do_resultado,data_de_liberacao,resultado_original,cidade_norm,...,apoio_cidade,gliese_uf,dasa_uf,apoio_uf,cep_municipio,cep_uf,cep_municipio_dasa,cep_uf_dasa,cidade_tratado,uf_tratado
6158,1002751831,2022-07-13 10:33:46+00:00,1984-02-03,38,FEMININO,DETECCAO QUALITATIVA DE CORONAVIRUS (2019-NCOV),2022-07-14 00:37:50+00:00,2022-07-14 01:13:35+00:00,NDT,SAOPAULO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35,1002751831,2022-10-24 10:02:24+00:00,1984-02-03,38,FEMININO,DETECCAO QUALITATIVA DE CORONAVIRUS (2019-NCOV),2022-10-25 07:43:50+00:00,NaN,NaN,SAOPAULO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_combined.query("test_id == '279800084151'")

,lab_id,test_id,test_kit,patient_id,sample_id,state,location,date_testing,epiweek,age,...,Ct_RDRP,geneS_detection,META_test_result,RINO_test_result,PARA_test_result,ADENO_test_result,BOCA_test_result,COVS_test_result,ENTERO_test_result,BAC_test_result
31584,DASA,279800084151,test_4,NaN,7b1859876e805cc6,Distrito Federal,NaN,2022-03-07,2022-03-12,43,...,NaN,NaN,NT,NT,NT,NT,NT,NT,NT,NT
31585,DASA,279800084151,test_4,NaN,dcb850bb21133ebf,Distrito Federal,BRASILIA,2022-03-07,2022-03-12,43,...,NaN,NaN,NT,NT,NT,NT,NT,NT,NT,NT


### Ids

Verificando se todos os Ids originais etão presentes no arquivo final

In [ ]:
set_test_ids_final_df = set(df_combined['test_id'])
set_test_ids_raw_data = set(df_full_raw['CODIGO REQUISICAO'])

NameError: name 'df_full_raw' is not defined

In [ ]:
# Testar se todos os test_ids da planilha original estão na planilha final
# deve retornar um set vazio
set_test_ids_raw_data.difference(set_test_ids_final_df)

{'1200085178_100',
 '1410231927_100',
 '1421499943_100',
 '1421512238_100',
 '1421512252_100',
 '1421524508_100',
 '1421542791_100',
 '1421567484_600',
 '1421573025_900',
 '1421574120_100',
 '1421580321_100',
 '1421580345_100',
 '1421580359_600',
 '1421580372_100',
 '1421580414_700',
 '1421590360_100',
 '1421608991_700',
 '1421618661_100',
 '1421623648_100',
 '1421627267_100',
 '1421641149_200',
 '1421661054_100',
 '1421747497_400',
 '1450064023_100',
 '1450064862_100',
 '1600683873_100',
 '1600708233_100',
 '1600709104_100',
 '1600712587_100',
 '1650441468_100',
 '1650447361_100',
 '1650451652_100',
 '1660378198_100',
 '1720397744_100',
 '1860281386_100',
 '1870163611_100',
 '2230148082_100',
 '2240159856_100',
 '2240169073_100',
 '2240170521_100',
 '2240193197_100',
 '2240197205_100',
 '2310036024_100',
 '2320523931_100',
 '2320536507_100',
 '2320542723_100',
 '2320550391_100',
 '2320551123_100',
 '2320551273_100',
 '2320562210_100',
 '2320582241_100',
 '2350037816_100',
 '2350041791

In [ ]:
# Verificar se há test_ids na planilha final que não estão na planilha original
# deve retornar um set vazio
set_test_ids_final_df.difference(set_test_ids_raw_data)

set()

Verificando se todos os patógenos de um test_kit estão sendo testados para todos os test_id

In [ ]:
PATHOGENS_TESTS = {
    'panel_21':{
        # PAINCOVI
        'FLUA_test_result',
        'FLUB_test_result',
        'VSR_test_result',
        'SC2_test_result',
        'META_test_result',
        'RINO_test_result',
        'PARA_test_result',
        'ADENO_test_result',
        'COVS_test_result',
        'BAC_test_result',
    },

    'panel_24':{
        # RESPIRA
        'FLUA_test_result',
        'FLUB_test_result',
        'VSR_test_result',
        'META_test_result',
        'RINO_test_result',
        'PARA_test_result',
        'ADENO_test_result',
        'BOCA_test_result',
        'COVS_test_result',
        'ENTERO_test_result',
        'BAC_test_result',
    },

    'flu_antigen':{
        'FLUA_test_result',
        'FLUB_test_result',
    },

    'covid_antigen':{
        'SC2_test_result',
    },

    'covid_pcr':{
        'SC2_test_result',
    },
}

In [ ]:
# Mapeando Pos e Neg para Tes
df_combined_test_pathogen_non_nt_on_kit = (
    df_combined
    .replace(('Pos', 'Neg'),  'Tes')
)

In [ ]:
for pathogen, test_columns in PATHOGENS_TESTS.items():
    print(pathogen)
    print(
        df_combined_test_pathogen_non_nt_on_kit
        .query("test_kit == @pathogen")
        [list(test_columns)]
        .drop_duplicates().T
        # Deve resultar em apenas uma linha completa por 'Tes'
    )

    print('\n\n')

panel_21
Empty DataFrame
Columns: []
Index: [FLUB_test_result, PARA_test_result, BAC_test_result, COVS_test_result, META_test_result, VSR_test_result, ADENO_test_result, RINO_test_result, FLUA_test_result, SC2_test_result]



panel_24
Empty DataFrame
Columns: []
Index: [FLUB_test_result, PARA_test_result, ENTERO_test_result, BAC_test_result, COVS_test_result, META_test_result, VSR_test_result, ADENO_test_result, BOCA_test_result, RINO_test_result, FLUA_test_result]



flu_antigen
Empty DataFrame
Columns: []
Index: [FLUB_test_result, FLUA_test_result]



covid_antigen
Empty DataFrame
Columns: []
Index: [SC2_test_result]



covid_pcr
                1929
SC2_test_result  Tes





Verificando resultados de alguns testes

In [ ]:
import numpy as np

In [ ]:
np.random.seed(214)

ids = np.random.choice(df_combined['test_id'], 100)
test_columns = [col for col in df_combined.columns if col.endswith('_result')]

In [ ]:
current_id=50

In [ ]:
current_id = current_id+1
id = ids[current_id]
print( df_combined.query("test_id == @id")[['test_id', 'test_kit'] + test_columns].T )
print( df_hla_covid.query(f"Pedido == '{id}'") [['Pedido'] + [col for col in df_hla_covid.columns if col.startswith('Vírus')]].T )
print( df_hla_ctn.query(f"Pedido == '{id}'") [['Pedido', 'Resultado']].T )
print( df_hla_flu.query(f"Pedido == '{id}'") [['Pedido']+ [col for col in df_hla_flu.columns if col.startswith('VIRUS') or col.startswith('BAC')]].T )
print( df_hla_ph4.query(f"Pedido == '{id}'") [['Pedido']+ [col for col in df_hla_ph4.columns if col.startswith('VIRUS') or col.startswith('BAC')]].T )

                         7665
test_id               2262865
test_kit            covid_pcr
FLUA_test_result           NT
FLUB_test_result           NT
VSR_test_result            NT
SC2_test_result           Neg
META_test_result           NT
RINO_test_result           NT
PARA_test_result           NT
ADENO_test_result          NT
BOCA_test_result           NT
COVS_test_result           NT
ENTERO_test_result         NT
BAC_test_result            NT
Empty DataFrame
Columns: []
Index: [Pedido, Vírus Influenza A, Vírus Influenza B, Vírus Sincicial Respiratório A/B]
                   1912
Pedido          2262865
Resultado  Inconclusivo
Empty DataFrame
Columns: []
Index: [Pedido, VIRUS_Influenza A, VIRUS_Influenza H1N1, VIRUS_Influenza H3, VIRUS_Influenza B, VIRUS_Metapneumovírus, VIRUS_Sincicial A, VIRUS_Sincicial B, VIRUS_Rinovírus, VIRUS_Parainfluenza 1, VIRUS_Parainfluenza 2, VIRUS_Parainfluenza 3, VIRUS_Parainfluenza 4, VIRUS_Adenovirus, VIRUS_Bocavirus, VIRUS_CoV-229E, VIRUS_CoV-HKU, VI

Verificando resultados de alguns testes - Fltrando por test_kit

In [ ]:
np.random.seed(214)

ids = np.random.choice(df_combined.query("test_kit not in ('covid_pcr', 'covid_antigen')")['test_id'], 100)
test_columns = [col for col in df_combined.columns if col.endswith('_result')]

In [ ]:
current_id=50

In [ ]:
current_id = current_id+1
id = ids[current_id]
print( df_combined.query("test_id == @id")[['test_id', 'test_kit'] + test_columns].T )
print( df_hla_covid.query(f"Pedido == '{id}'") [['Pedido'] + [col for col in df_hla_covid.columns if col.startswith(('Vírus', 'Coron'))]].T )
print( df_hla_flu.query(f"Pedido == '{id}'") [['Pedido']+ [col for col in df_hla_flu.columns if col.startswith('VIRUS') or col.startswith('BAC')]].T )
print( df_hla_ph4.query(f"Pedido == '{id}'") [['Pedido']+ [col for col in df_hla_ph4.columns if col.startswith('VIRUS') or col.startswith('BAC')]].T )

                      50943
test_id             2354190
test_kit             test_4
FLUA_test_result        Neg
FLUB_test_result        Neg
VSR_test_result         Pos
SC2_test_result         Neg
META_test_result         NT
RINO_test_result         NT
PARA_test_result         NT
ADENO_test_result        NT
BOCA_test_result         NT
COVS_test_result         NT
ENTERO_test_result       NT
BAC_test_result          NT
                                             18
Pedido                                  2354190
Vírus Influenza A                 Não Detectado
Vírus Influenza B                 Não Detectado
Vírus Sincicial Respiratório A/B      Detectado
Coronavírus SARS-CoV-2            Não Detectado
Empty DataFrame
Columns: []
Index: [Pedido, VIRUS_Influenza A, VIRUS_Influenza H1N1, VIRUS_Influenza H3, VIRUS_Influenza B, VIRUS_Metapneumovírus, VIRUS_Sincicial A, VIRUS_Sincicial B, VIRUS_Rinovírus, VIRUS_Parainfluenza 1, VIRUS_Parainfluenza 2, VIRUS_Parainfluenza 3, VIRUS_Parainfluenza 4